## 课程实验

# D04：湖表与内部表关联

Data Warehousing with Apache Doris · Level 1

[讲义](course.md) · [课程入口](../README.md)


## 环境要求与状态

这是需要外部环境的候选 Lab，首版不提供 Iceberg 服务部署。讲师需预置包含 datasets/orders.json 全部字段的 Iceberg orders 表，并通过 Doris Catalog 可查询。设置 DW_ICEBERG_ORDERS=catalog.database.table；没有该环境时应明确记录未执行，不用内部表冒充湖表。


In [ ]:
import os
from dw_course.runtime import identifier
source_parts = os.environ["DW_ICEBERG_ORDERS"].split(".")
if len(source_parts) != 3:
    raise ValueError("Expected catalog.database.table")
source = ".".join(identifier(part) for part in source_parts)
from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows

lab = WarehouseLab()
print("Course database:", lab.database)



## 1. 直查湖表

同一份 10 笔订单应有 1400.00 金额。这里没有导入数据；Catalog 提供外部表元数据。


In [ ]:
expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {source}"), [(10,"1400.00")])
print(lab.query(f"EXPLAIN SELECT * FROM {source} WHERE order_id = 1001"))


## 2. 与内部客户表关联

每个订单客户对应一个客户行；关联后数量不能扩大。仅重建 d04_customers 与 d04_orders。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d04_customers")
lab.execute('CREATE TABLE d04_customers (customer_id BIGINT, region VARCHAR(16)) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.insert("d04_customers", ["customer_id","region"], [(r["customer_id"],r["region"]) for r in fixture("orders.json")])
expect(lab.query(f"SELECT COUNT(*), SUM(o.order_amount) FROM {source} o JOIN d04_customers c ON o.customer_id=c.customer_id"), [(10,"1400.00")])
lab.execute("DROP TABLE IF EXISTS d04_orders")
ddl = order_ddl("d04_orders")
print(ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO d04_orders ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM {source}")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM d04_orders"), [(10,"1400.00")])
lab.close()


## 完成与边界

记录 Catalog 类型和外部服务版本、原表标识、结果与计划。独立 Parquet 的 S3 TVF 实验仍待补；读取成功不代表外部写入、Schema 演进或性能 SLA 已验证。
